In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import squarify
import seaborn as sns
import yaml
from utils import apply_rehydratation, read_stats
from ipywidgets import FloatSlider, VBox, HBox, Output, Button, Layout, Label
from IPython.display import display


In [2]:
# Load the YAML file
with open("/home/ogouvert/OpenLLM-BPI-Training/data/tokenization/datasets_to_tokenize.yaml", "r") as file:
    data = yaml.safe_load(file)

metadata = pd.DataFrame(data['datasets'])

In [4]:
df = read_stats("/media/storage0/ogouvert/OpenLLM-BPI-output/data/tokens_ablation/stats")
df = df.merge(metadata, on='name', how='left')
df = apply_rehydratation(df, 'total_tokens')
df['total_tokens_rehydrated'] = df['total_tokens_rehydrated'] / 10**9

# Rename and group
df = df[['name_pro', 'category', 'language', 'total_tokens_rehydrated']]
df = df.rename(columns={'name_pro': 'name', 'total_tokens_rehydrated': 'tokens'})
df = df.groupby(['name', 'category', 'language'], as_index=False).sum()
df['upsampling'] = 1
df['tokens_estimation (B)'] = df['tokens'] * 20

In [5]:
def plot_treemap(df, label_column):
    df = df[df['upsampled_tokens'] > 0]
    df = df.sort_values(by='upsampled_tokens', ascending=False)
    plt.figure(figsize=(10, 5))
    squarify.plot(sizes = df['upsampled_tokens'], label = df[label_column],
                pad = 0.05, alpha=0.7,
                text_kwargs = {'fontsize': 8, 'color': 'black'},
                color = sns.color_palette("rocket", len(df)))

    plt.axis("off")
    plt.show()

In [ ]:
# Save original upsampling values
original_values = df['upsampling'].tolist()

# Create labeled sliders (labels above sliders, no truncation)
slider_widgets = []
sliders = []

for i in range(len(df)):
    name = df.loc[i, 'name']
    slider = FloatSlider(
        value=original_values[i],
        min=0.0,
        max=5.0,
        step=0.01,
        continuous_update=False,
        layout=Layout(width='95%')
    )
    label = Label(value=name, layout=Layout(width='95%'))
    sliders.append(slider)
    slider_widgets.append(VBox([label, slider]))

# Output area for plots
out = Output(layout=Layout(flex='1 1 auto', width='auto', height='auto', overflow='visible'))

# Output area for df display at the bottom left
df_out = Output(layout=Layout(width='100%', height='auto', display='flex', align_items='flex-start', justify_content='flex-start'))

def plot():
    with out:
        out.clear_output(wait=True)
        df['upsampling'] = [s.value for s in sliders]
        df['upsampled_tokens'] = df['tokens'] * df['upsampling']
        df['weight'] = df['upsampled_tokens'] / df['upsampled_tokens'].sum()
        df['seen_tokens (B)'] = df['weight'] * 3 * 10**3  # in billion
        df['epochs'] = df['seen_tokens (B)'] / df['tokens_estimation (B)']

        # All        
        plot_treemap(df, label_column='name')

        # Categories
        cat_df = df.groupby('category').agg({'upsampled_tokens': 'sum'}).reset_index()
        plot_treemap(cat_df, label_column='category')

        # Languages
        lan_df = df.groupby('language').agg({'upsampled_tokens': 'sum'}).reset_index()
        plot_treemap(lan_df, label_column='language')

    with df_out:
        df_out.clear_output(wait=True)
        print(df[['name', 'category', 'language', 'seen_tokens (B)', 'epochs']])

# Attach observers to update on slider value change
for slider in sliders:
    slider.observe(lambda change: plot(), names='value')

# Reset button
reset_button = Button(description='Reset', button_style='warning', layout=Layout(width='95%'))

def reset_sliders(b):
    for i, slider in enumerate(sliders):
        slider.value = original_values[i]

reset_button.on_click(reset_sliders)

# Output area with flexible layout
out = Output(layout=Layout(flex='1 1 auto', width='auto'))

# Assemble layout: sliders and reset button on the left, plots on the right
slider_panel = VBox([reset_button] + slider_widgets, layout=Layout(width='20%'))
figures_panel = VBox([out], layout=Layout(width='80%'))

# Main layout with df display at the bottom
main_layout = VBox([HBox([slider_panel, figures_panel]), df_out], layout=Layout(width='100%'))

# Initial plot
plot()

# Display UI
display(main_layout)